# Fase 3: Limpieza y preprocesamiento del dataset

En esta fase se realiza la limpieza y el preprocesamiento del dataset de citas médicas con el objetivo de dejar los datos en un formato consistente, reproducible y adecuado para las siguientes etapas del proyecto de Machine Learning.

## Objetivo de la fase

El objetivo de esta fase es preparar el dataset de forma profesional para su uso en etapas posteriores del proyecto.

Las actividades principales son:

- convertir correctamente las variables de fecha;
- corregir o filtrar edades inválidas;
- revisar posibles outliers;
- definir correctamente la variable objetivo;
- transformar variables categóricas;
- dejar un flujo de limpieza claro y reproducible.

In [1]:
import pandas as pd
import numpy as np

In [2]:
path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016.csv"
df = pd.read_csv(path)

print("Dataset cargado correctamente")
print(df.shape)
df.head()

Dataset cargado correctamente
(110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## Copia de trabajo

Se crea una copia del dataset original para aplicar el preprocesamiento sin modificar directamente la versión cruda de los datos.

In [3]:
df_clean = df.copy()

print("Copia de trabajo creada")
print(df_clean.shape)

Copia de trabajo creada
(110527, 14)


## Revisión inicial de tipos de datos

Antes de limpiar el dataset, se verifica el tipo de dato actual de cada columna.

In [4]:
df_clean.dtypes

PatientId         float64
AppointmentID       int64
Gender                str
ScheduledDay          str
AppointmentDay        str
Age                 int64
Neighbourhood         str
Scholarship         int64
Hipertension        int64
Diabetes            int64
Alcoholism          int64
Handcap             int64
SMS_received        int64
No-show               str
dtype: object

## Conversión de variables temporales

Las columnas `ScheduledDay` y `AppointmentDay` se convierten a formato datetime para permitir su manipulación posterior de forma consistente.

In [5]:
df_clean['ScheduledDay'] = pd.to_datetime(df_clean['ScheduledDay'], errors='coerce')
df_clean['AppointmentDay'] = pd.to_datetime(df_clean['AppointmentDay'], errors='coerce')

df_clean[['ScheduledDay', 'AppointmentDay']].head()

,ScheduledDay,AppointmentDay
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00


In [6]:
print("Nulos en ScheduledDay después de convertir:", df_clean['ScheduledDay'].isnull().sum())
print("Nulos en AppointmentDay después de convertir:", df_clean['AppointmentDay'].isnull().sum())

df_clean[['ScheduledDay', 'AppointmentDay']].dtypes

Nulos en ScheduledDay después de convertir: 0
Nulos en AppointmentDay después de convertir: 0


ScheduledDay      datetime64[us, UTC]
AppointmentDay    datetime64[us, UTC]
dtype: object

## Transformación de la variable objetivo

La variable `No-show` se convierte a formato numérico para facilitar su uso posterior en modelos de Machine Learning.

Convención utilizada:
- 0 = asistió
- 1 = no asistió

In [7]:
df_clean['No_show'] = df_clean['No-show'].map({'No': 0, 'Yes': 1})

df_clean[['No-show', 'No_show']].head()

,No-show,No_show
0,No,0
1,No,0
2,No,0
3,No,0
4,No,0


In [8]:
print(df_clean['No_show'].value_counts(dropna=False))
print("Nulos en No_show:", df_clean['No_show'].isnull().sum())

No_show
0    88208
1    22319
Name: count, dtype: int64
Nulos en No_show: 0


## Revisión y tratamiento de edades inválidas

Se revisa la variable `Age` para identificar registros con edades negativas o valores poco realistas.

In [9]:
print("Edad mínima:", df_clean['Age'].min())
print("Edad máxima:", df_clean['Age'].max())

print("Edades menores a 0:", (df_clean['Age'] < 0).sum())
print("Edades mayores a 100:", (df_clean['Age'] > 100).sum())

Edad mínima: -1
Edad máxima: 115
Edades menores a 0: 1
Edades mayores a 100: 7


## Criterio de limpieza para edades

En esta fase se eliminarán los registros con edades negativas, ya que representan datos inválidos.

Las edades mayores a 100 años no se eliminarán automáticamente, ya que podrían corresponder a casos reales y primero se consideran como valores extremos a revisar.

In [10]:
rows_before = df_clean.shape[0]

df_clean = df_clean[df_clean['Age'] >= 0].copy()

rows_after = df_clean.shape[0]
rows_removed = rows_before - rows_after

print("Filas antes:", rows_before)
print("Filas después:", rows_after)
print("Filas eliminadas por edad negativa:", rows_removed)

Filas antes: 110527
Filas después: 110526
Filas eliminadas por edad negativa: 1


## Revisión de valores extremos

Se analizan variables numéricas para detectar posibles outliers que deban documentarse antes del modelado.

In [11]:
numeric_cols = ['Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received']
df_clean[numeric_cols].describe()

,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,110526.000000,110526.000000,110526.000000,110526.000000,110526.000000,110526.000000,110526.000000
mean,37.089219,0.098266,0.197248,0.071865,0.030400,0.022248,0.321029
std,23.110026,0.297676,0.397923,0.258266,0.171686,0.161543,0.466874
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


In [12]:
Q1 = df_clean['Age'].quantile(0.25)
Q3 = df_clean['Age'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", lower_bound)
print("Límite superior:", upper_bound)

age_outliers = df_clean[(df_clean['Age'] < lower_bound) | (df_clean['Age'] > upper_bound)]
print("Cantidad de posibles outliers en Age:", age_outliers.shape[0])

Q1: 18.0
Q3: 55.0
IQR: 37.0
Límite inferior: -37.5
Límite superior: 110.5
Cantidad de posibles outliers en Age: 5


## Interpretación de outliers

Los valores extremos detectados en `Age` se documentan, pero no se eliminan automáticamente en esta fase, salvo los casos claramente inválidos como edades negativas.

Esto permite mantener trazabilidad y evitar perder información potencialmente útil.

## Conversión de variables categóricas

Se transforman variables categóricas a un formato más consistente para etapas posteriores.

En esta fase:
- `Gender` se convertirá a variable binaria;
- `Neighbourhood` se conservará como categórica para una transformación posterior más adecuada;
- `No-show` original se conservará por trazabilidad, pero `No_show` será la variable objetivo final.

In [13]:
df_clean['Gender'] = df_clean['Gender'].map({'F': 0, 'M': 1})

print(df_clean['Gender'].value_counts(dropna=False))

Gender
0    71839
1    38687
Name: count, dtype: int64


In [14]:
print("Valores únicos de Gender:", df_clean['Gender'].unique())
print("Valores únicos de Scholarship:", sorted(df_clean['Scholarship'].unique()))
print("Valores únicos de Hipertension:", sorted(df_clean['Hipertension'].unique()))
print("Valores únicos de Diabetes:", sorted(df_clean['Diabetes'].unique()))
print("Valores únicos de Alcoholism:", sorted(df_clean['Alcoholism'].unique()))
print("Valores únicos de Handcap:", sorted(df_clean['Handcap'].unique()))
print("Valores únicos de SMS_received:", sorted(df_clean['SMS_received'].unique()))

Valores únicos de Gender: [0 1]
Valores únicos de Scholarship: [np.int64(0), np.int64(1)]
Valores únicos de Hipertension: [np.int64(0), np.int64(1)]
Valores únicos de Diabetes: [np.int64(0), np.int64(1)]
Valores únicos de Alcoholism: [np.int64(0), np.int64(1)]
Valores únicos de Handcap: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Valores únicos de SMS_received: [np.int64(0), np.int64(1)]


## Eliminación de columnas identificadoras

Las columnas `PatientId`, `AppointmentID` y `No-show` se eliminan del dataset preprocesado final:

- `PatientId` y `AppointmentID` son identificadores;
- `No-show` se reemplaza por la versión numérica `No_show`.

In [15]:
df_clean = df_clean.drop(columns=['PatientId', 'AppointmentID', 'No-show'])

print(df_clean.shape)
df_clean.head()

(110526, 12)


,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No_show
0,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,0
1,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,0
2,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,0
3,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,0
4,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,0


## Revisión final del dataset preprocesado

Se verifica la estructura final del dataset después del proceso de limpieza y preprocesamiento.

In [16]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 110526 entries, 0 to 110526
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   Gender          110526 non-null  int64              
 1   ScheduledDay    110526 non-null  datetime64[us, UTC]
 2   AppointmentDay  110526 non-null  datetime64[us, UTC]
 3   Age             110526 non-null  int64              
 4   Neighbourhood   110526 non-null  str                
 5   Scholarship     110526 non-null  int64              
 6   Hipertension    110526 non-null  int64              
 7   Diabetes        110526 non-null  int64              
 8   Alcoholism      110526 non-null  int64              
 9   Handcap         110526 non-null  int64              
 10  SMS_received    110526 non-null  int64              
 11  No_show         110526 non-null  int64              
dtypes: datetime64[us, UTC](2), int64(9), str(1)
memory usage: 11.0 MB


In [17]:
df_clean.isnull().sum()

Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No_show           0
dtype: int64

In [18]:
df_clean.head()

,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No_show
0,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,0
1,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,0
2,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,0
3,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,0
4,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,0


## Pipeline de limpieza aplicado

Durante esta fase se aplicó el siguiente flujo de preprocesamiento:

1. Carga del dataset original.
2. Creación de una copia de trabajo.
3. Conversión de `ScheduledDay` y `AppointmentDay` a formato datetime.
4. Transformación de la variable objetivo `No-show` a `No_show`.
5. Identificación y eliminación de edades negativas.
6. Revisión de outliers en la variable `Age`.
7. Conversión de `Gender` a formato numérico.
8. Eliminación de columnas identificadoras.
9. Validación final del dataset limpio.

Este flujo permite que el preprocesamiento sea claro, reproducible y reutilizable en siguientes fases del proyecto.

In [19]:
output_path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-clean.csv"

df_clean.to_csv(output_path, index=False)

print("Dataset limpio guardado en:")
print(output_path)

Dataset limpio guardado en:
C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-clean.csv


## Conclusión de la Fase 3

El dataset fue limpiado y preparado mediante un flujo reproducible que incluye conversión de variables temporales, transformación de la variable objetivo, tratamiento de valores inválidos y eliminación de columnas no relevantes.

El resultado es un dataset consistente, sin valores nulos y con variables en formatos adecuados para el modelado.

Este proceso establece una base sólida para la siguiente fase de ingeniería de características, donde se generarán nuevas variables que permitan mejorar la capacidad predictiva del modelo.